# SCR.data | Monitoramento Temporal de Variações Atípicas

**Ferramenta experimental de análise de dados públicos agregados do Sistema de Informações de Créditos (SCR) do Banco Central do Brasil.**

Este notebook consulta os arquivos públicos do SCR.data, prepara uma base mensal de pessoas jurídicas, aplica critérios exploratórios de movimentos compensatórios e mudanças de concentração, e produz tabelas, gráficos e arquivos para auditoria e reprodução.

**Interpretação:** os alertas indicam comportamentos estatísticos que merecem investigação. Não comprovam erros, reclassificação dos mesmos contratos ou falhas de instituições financeiras individuais.

**Período padrão:** janeiro de 2025 a julho de 2026. **Ambiente:** Python / Google Colab. Os parâmetros podem ser alterados na seção 1.


## 1. Configuração e dependências

In [ ]:
%pip -q install pandas numpy matplotlib requests pyarrow openpyxl

from pathlib import Path
import io, re, zipfile, hashlib, json, unicodedata, time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ANO_INICIAL, ANO_FINAL = 2025, 2026
MES_INICIAL, MES_FINAL = '2025-01', '2026-07'  # altere para ampliar o horizonte
PASTA = Path('scr_monitoramento')
DADOS, SAIDA = PASTA / 'dados', PASTA / 'resultados'
DADOS.mkdir(parents=True, exist_ok=True); SAIDA.mkdir(parents=True, exist_ok=True)
BASE_URL = 'https://www.bcb.gov.br/pda/desig/scrdata_{ano}.zip'
LIMIAR_VALOR = 100_000_000  # R$ 100 milhões
LIMIAR_COMPENSACAO = 95.0   # % de equilíbrio entre os dois movimentos
LIMIAR_CONCENTRACAO = 80.0  # % da carteira na categoria predominante
LIMIAR_SALDO_CONCENTRACAO = 1_000_000_000
# Apenas para diagnóstico/sensibilidade: 90%, 95%, 98%
LIMIARES_SENSIBILIDADE = (90., 95., 98.)
print('Configuração pronta:', MES_INICIAL, 'a', MES_FINAL)

## 2. Download oficial e inventário dos arquivos

O download utiliza cache e registra SHA-256, tamanho e URL. Caso o Banco Central altere o layout, a execução será interrompida com mensagem explícita.

In [ ]:
def baixar_zip(ano):
    destino = DADOS / f'scrdata_{ano}.zip'
    url = BASE_URL.format(ano=ano)
    if not destino.exists():
        print(f'Baixando {url} (pode demorar)...')
        with requests.get(url, stream=True, timeout=(30, 300), headers={'User-Agent':'SCRdata-Temporal-Monitor/1.0'}) as r:
            r.raise_for_status()
            with open(destino, 'wb') as f:
                for parte in r.iter_content(chunk_size=2**20):
                    if parte: f.write(parte)
    if not zipfile.is_zipfile(destino):
        raise ValueError(f'O arquivo {destino} não é um ZIP válido. Exclua o cache e tente novamente.')
    sha = hashlib.sha256()
    with open(destino,'rb') as f:
        for parte in iter(lambda:f.read(2**20),b''): sha.update(parte)
    with zipfile.ZipFile(destino) as z:
        arquivos = [x for x in z.namelist() if x.lower().endswith('.csv') and not x.endswith('/')]
    if not arquivos: raise ValueError(f'Nenhum CSV no ZIP {destino}; verifique a versão do SCR.data.')
    return {'ano':ano,'url':url,'zip':str(destino),'sha256':sha.hexdigest(),
            'bytes':destino.stat().st_size,'csvs':len(arquivos)}

inventario = pd.DataFrame([baixar_zip(a) for a in range(ANO_INICIAL, ANO_FINAL+1)])
display(inventario)
inventario.to_csv(SAIDA/'inventario_fontes.csv',index=False,encoding='utf-8-sig')

## 3. Leitura dos CSVs e preparação da base PJ

Leitura em blocos, sem extrair todos os CSVs no disco. A data do mês é validada pelo nome do arquivo e/ou coluna `data_base`. Os saldos mantêm sua unidade original (R$).

In [ ]:
DIMENSOES = ['modalidade','submodalidade','segmento','cnae_ocupacao','uf','origem','indexador','porte']
NECESSARIAS = DIMENSOES + ['cliente','carteira_ativa']

def normalizar_coluna(x):
    x=unicodedata.normalize('NFKD',str(x).replace('\ufeff','').replace('ï»¿','').lower())
    x=''.join(c for c in x if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]+','_',x).strip('_')

def mes_nome(nome):
    m=re.search(r'(20\d{2})[_-]?(0[1-9]|1[0-2])(?=\D|$)',Path(nome).name)
    return pd.Period(f'{m.group(1)}-{m.group(2)}',freq='M') if m else None

def converter_valores(serie):
    if pd.api.types.is_numeric_dtype(serie): return pd.to_numeric(serie,errors='coerce')
    s=serie.astype('string').str.strip()
    br=pd.to_numeric(s.str.replace('.','',regex=False).str.replace(',','.',regex=False),errors='coerce')
    en=pd.to_numeric(s,errors='coerce')
    return br.where(s.str.contains(',',regex=False,na=False),en).fillna(br)

def ler_membro(z, membro, ano, blocos=100_000):
    with z.open(membro) as f:
        cab=f.readline()
    enc='utf-8-sig'
    try: cab.decode(enc)
    except UnicodeDecodeError: enc='latin1'
    sep=';' if cab.count(b';') >= cab.count(b',') else ','
    periodo_nome=mes_nome(membro)
    with z.open(membro) as f:
        leitor=pd.read_csv(f,sep=sep,encoding=enc,chunksize=blocos,dtype=str,low_memory=False)
        for bloco in leitor:
            bloco.columns=[normalizar_coluna(c) for c in bloco.columns]
            faltantes=[c for c in NECESSARIAS if c not in bloco.columns]
            if faltantes:
                raise ValueError(f'Layout incompatível no arquivo {membro}: faltam {faltantes}. Encontradas: {list(bloco.columns)}')
            pj=bloco[bloco['cliente'].astype('string').str.strip().str.upper().eq('PJ')].copy()
            if pj.empty: continue
            if 'data_base' in pj:
                datas=pd.to_datetime(pj['data_base'],errors='coerce',format='%Y-%m-%d')
                if datas.isna().all(): datas=pd.to_datetime(pj['data_base'],errors='coerce',dayfirst=True)
                periodos=datas.dt.to_period('M')
                if periodo_nome is not None and periodos.notna().any() and (periodos.dropna()!=periodo_nome).any():
                    raise ValueError(f'Data divergente do nome: {membro}')
                pj['periodo']=periodos.astype('string').fillna(str(periodo_nome) if periodo_nome else '')
            elif periodo_nome is not None: pj['periodo']=str(periodo_nome)
            else: raise ValueError(f'Não foi possível identificar o período de {membro}')
            pj=pj[pj['periodo'].between(MES_INICIAL,MES_FINAL)]
            if pj.empty: continue
            for c in DIMENSOES: pj[c]=pj[c].astype('string').str.strip().fillna('Não informado')
            pj['carteira_ativa']=converter_valores(pj['carteira_ativa'])
            if pj['carteira_ativa'].isna().any():
                raise ValueError(f'Saldo não numérico no arquivo {membro}: {pj["carteira_ativa"].isna().sum()} linhas')
            yield pj[DIMENSOES+['periodo','carteira_ativa']]

mensais=[]; processados=[]
for registro in inventario.itertuples():
    with zipfile.ZipFile(registro.zip) as z:
        membros=sorted(m for m in z.namelist() if m.lower().endswith('.csv'))
        for membro in membros:
            periodo=mes_nome(membro)
            if periodo is not None and not (MES_INICIAL<=str(periodo)<=MES_FINAL): continue
            parciais=[]; linhas=0
            for bloco in ler_membro(z,membro,registro.ano):
                linhas+=len(bloco)
                parciais.append(bloco.groupby(DIMENSOES+['periodo'],dropna=False,observed=True,as_index=False)['carteira_ativa'].sum())
            if parciais:
                mensal=pd.concat(parciais,ignore_index=True).groupby(DIMENSOES+['periodo'],dropna=False,observed=True,as_index=False)['carteira_ativa'].sum()
                mensais.append(mensal)
                processados.append({'arquivo':membro,'ano':registro.ano,'linhas_pj':linhas,'periodos':','.join(sorted(mensal.periodo.unique()))})
                print('Processado:',membro,linhas,'linhas PJ')
if not mensais: raise RuntimeError('Nenhum registro PJ encontrado no período configurado.')
base_granular=pd.concat(mensais,ignore_index=True).groupby(DIMENSOES+['periodo'],dropna=False,observed=True,as_index=False)['carteira_ativa'].sum()
base_granular['periodo']=pd.PeriodIndex(base_granular['periodo'],freq='M')
meses_presentes=sorted(base_granular['periodo'].unique())
print('Linhas agregadas:',len(base_granular),'| meses:',len(meses_presentes),'|',meses_presentes[0],meses_presentes[-1])
base_granular.to_parquet(SAIDA/'base_granular_PJ.parquet',index=False)
pd.DataFrame(processados).to_csv(SAIDA/'arquivos_processados.csv',index=False,encoding='utf-8-sig')

## 4. Detecção de movimentos compensatórios entre categorias

**Regra:** em recorte com todas as outras dimensões fixas, procurar categorias com variações mensais opostas, valor compensado mínimo e taxa `200 × min(|ΔA|,|ΔB|) / (|ΔA|+|ΔB|)`. Somente se comparam saldos **publicados em ambos os meses consecutivos**; ausências não viram zeros. Não se inferem migrações individuais.

In [ ]:
def movimentos_compensatorios(base, categoria, limiar_valor=LIMIAR_VALOR, limiar_pct=LIMIAR_COMPENSACAO):
    fixas=[c for c in DIMENSOES if c!=categoria]
    g=(base.groupby(fixas+[categoria,'periodo'],dropna=False,observed=True,as_index=False)['carteira_ativa'].sum()
       .sort_values(fixas+[categoria,'periodo']))
    grp=g.groupby(fixas+[categoria],dropna=False,observed=True,sort=False)
    anterior=grp['carteira_ativa'].shift()
    mes_ant=grp['periodo'].shift()
    atual_ord=g['periodo'].astype('int64')
    anterior_ord=mes_ant.fillna(g['periodo']).astype('int64')
    g['variacao']=(g['carteira_ativa']-anterior).where(mes_ant.notna() & ((atual_ord-anterior_ord)==1))
    g=g[g['variacao'].abs()>=limiar_valor].copy()
    if g.empty: return pd.DataFrame()
    pos=g[g.variacao>0].rename(columns={categoria:'categoria_aumento','variacao':'aumento'})
    neg=g[g.variacao<0].rename(columns={categoria:'categoria_reducao','variacao':'reducao'})
    pares=pos[fixas+['periodo','categoria_aumento','aumento']].merge(
        neg[fixas+['periodo','categoria_reducao','reducao']],on=fixas+['periodo'],how='inner')
    if pares.empty: return pares
    pares['reducao']=pares['reducao'].abs()
    pares['valor_compensado']=np.minimum(pares['aumento'],pares['reducao'])
    pares['taxa_compensacao_pct']=200*pares['valor_compensado']/(pares['aumento']+pares['reducao'])
    pares['tipo_alerta']='Redistribuição aparente: '+categoria
    return pares[(pares.valor_compensado>=limiar_valor)&(pares.taxa_compensacao_pct>=limiar_pct)].sort_values('valor_compensado',ascending=False).reset_index(drop=True)

alertas_porte=movimentos_compensatorios(base_granular,'porte')
alertas_indexador=movimentos_compensatorios(base_granular,'indexador')
print('Porte:',len(alertas_porte),'| Indexador:',len(alertas_indexador))
display(alertas_porte.head(10)); display(alertas_indexador.head(10))
for nome,df in [('alertas_porte',alertas_porte),('alertas_indexador',alertas_indexador)]:
    df.to_csv(SAIDA/f'{nome}.csv',index=False,encoding='utf-8-sig')

## 5. Mudanças de concentração e reversões

Acompanhamento da categoria de porte com maior saldo no recorte; mudança apenas entre meses consecutivos. Reversão = A → B → A em três meses consecutivos, quando ambos os movimentos são alertas.

In [ ]:
FIXAS_PORTE=[c for c in DIMENSOES if c!='porte']
def detectar_concentracao(base):
    agg=base.groupby(FIXAS_PORTE+['periodo','porte'],dropna=False,observed=True,as_index=False)['carteira_ativa'].sum()
    totais=agg.groupby(FIXAS_PORTE+['periodo'],dropna=False,observed=True,as_index=False)['carteira_ativa'].sum().rename(columns={'carteira_ativa':'total'})
    top=(agg.sort_values(FIXAS_PORTE+['periodo','carteira_ativa','porte'],ascending=[True]*8+[False,True])
         .drop_duplicates(FIXAS_PORTE+['periodo']).merge(totais,on=FIXAS_PORTE+['periodo']))
    top['participacao_pct']=np.where(top.total>0,100*top.carteira_ativa/top.total,np.nan)
    top=top.sort_values(FIXAS_PORTE+['periodo']).reset_index(drop=True)
    gr=top.groupby(FIXAS_PORTE,dropna=False,observed=True,sort=False)
    top['porte_anterior']=gr.porte.shift(1)
    top['mes_anterior']=gr.periodo.shift(1)
    top['porte_2m']=gr.porte.shift(2)
    top['mes_2m']=gr.periodo.shift(2)
    top['consecutivo']=top.mes_anterior.notna() & ((top.periodo.astype('int64')-top.mes_anterior.fillna(top.periodo).astype('int64'))==1)
    top['alerta']=top.consecutivo & top.porte.ne(top.porte_anterior) & (top.carteira_ativa>=LIMIAR_SALDO_CONCENTRACAO) & (top.participacao_pct>=LIMIAR_CONCENTRACAO)
    anterior_alerta=top.groupby(FIXAS_PORTE,dropna=False,observed=True,sort=False).alerta.shift(1).fillna(False)
    top['reversao']=top.alerta & anterior_alerta & top.mes_2m.notna() & ((top.periodo.astype('int64')-top.mes_2m.fillna(top.periodo).astype('int64'))==2) & top.porte.eq(top.porte_2m)
    return top[top.alerta].copy()

alertas_concentracao=detectar_concentracao(base_granular)
print('Concentração:',len(alertas_concentracao),'| reversões:',int(alertas_concentracao.reversao.sum()))
display(alertas_concentracao.head(10))
alertas_concentracao.to_csv(SAIDA/'alertas_concentracao.csv',index=False,encoding='utf-8-sig')

## 6. Análise de sensibilidade e resumo

Contagens podem diferir do experimento histórico: aquele método de porte preenchia meses sem publicação com zero; este notebook **não** o faz. Resultados não são calibrados para atingir totais antigos.

In [ ]:
sensibilidade=[]
for limiar in LIMIARES_SENSIBILIDADE:
    for categoria in ('porte','indexador'):
        dados=movimentos_compensatorios(base_granular,categoria,limiar_pct=limiar)
        sensibilidade.append({'categoria':categoria,'compensacao_minima_pct':limiar,'alertas':len(dados)})
sensibilidade=pd.DataFrame(sensibilidade)
display(sensibilidade)
sensibilidade.to_csv(SAIDA/'sensibilidade.csv',index=False,encoding='utf-8-sig')
resumo=pd.DataFrame([{'metodo':'Porte','alertas':len(alertas_porte)},
                     {'metodo':'Indexador','alertas':len(alertas_indexador)},
                     {'metodo':'Concentração','alertas':len(alertas_concentracao)},
                     {'metodo':'Reversões (subconjunto)','alertas':int(alertas_concentracao.reversao.sum())}])
display(resumo)
resumo.to_csv(SAIDA/'resumo_alertas.csv',index=False,encoding='utf-8-sig')

## 7. Visualização exploratória por recorte

A tabela e o gráfico mostram **todas as categorias publicadas** da dimensão escolhida, preservando a visão de conjunto. Valores ausentes permanecem ausentes, sem imputação automática de zero. O recorte automático é apenas uma demonstração: os demais recortes podem ser consultados na seção 8.


In [ ]:
def visualizar_caso(base, categoria, filtros, inicio='2025-01', fim='2026-07', titulo=None):
    df=base.copy()
    for coluna,valor in filtros.items():
        if coluna not in DIMENSOES: raise KeyError(coluna)
        df=df[df[coluna].astype(str)==str(valor)]
    df=df[df.periodo.astype(str).between(inicio,fim)]
    if df.empty:
        print('Nenhuma linha encontrada. Confira a grafia das categorias no SCR.data.');return None
    serie=df.groupby([categoria,'periodo'],observed=True,dropna=False)['carteira_ativa'].sum().unstack('periodo')/1e9
    serie=serie.reindex(columns=pd.period_range(inicio,fim,freq='M'))
    display(serie.round(3).rename_axis(index=categoria,columns='mês'))
    ax=serie.T.plot(figsize=(12,5),marker='o',linewidth=2)
    ax.set(title=titulo or 'Saldo por categoria — SCR.data',xlabel='Mês',ylabel='Carteira ativa (R$ bilhões)')
    ax.grid(alpha=.2); plt.xticks(rotation=45);plt.tight_layout();plt.show()
    return serie

# Seleção AUTOMÁTICA do maior alerta de porte: demonstra o uso sem filtros manuais.
if not alertas_porte.empty:
    caso=alertas_porte.iloc[0]
    fixas={c:caso[c] for c in DIMENSOES if c!='porte'}
    mes=str(caso['periodo']); ini=str(max(pd.Period(MES_INICIAL),pd.Period(mes)-2)); fim=str(min(pd.Period(MES_FINAL),pd.Period(mes)+2))
    print('Maior alerta de porte:',mes,'|',caso['categoria_reducao'],'→',caso['categoria_aumento'])
    tabela_exemplo=visualizar_caso(base_granular,'porte',fixas,ini,fim,'Maior movimento compensatório entre portes')
else: print('Nenhum alerta de porte para os parâmetros atuais.')

## 8. Estudos de caso e visualização comparativa

Os quatro exemplos abaixo ilustram comportamentos identificados no conjunto público: variações entre portes, entre indexadores e reversão temporal. Cada estudo apresenta **todas as categorias disponíveis** no recorte fixado, tabela em R$ bilhões e gráfico temporal. A interpretação é exploratória; não há rastreabilidade contratual.


In [ ]:
# Recortes documentados para reprodução de exemplos de monitoramento temporal.
ESTUDOS_DE_CASO = {
    '01_variacao_porte_set2025': {
        'dimensao': 'porte',
        'titulo': 'Variação entre portes — setembro de 2025',
        'destaque': '2025-09',
        'filtros': {'modalidade':'Empréstimos','submodalidade':'Capital de giro com prazo de vencimento até 365 dias','segmento':'Banco','cnae_ocupacao':'Atividades financeiras, de seguros e serviços relacionados','uf':'SP','origem':'Sem destinação específica','indexador':'Prefixado'}
    },
    '02_variacao_indexador_jan2026': {
        'dimensao': 'indexador',
        'titulo': 'Variação entre indexadores — janeiro de 2026',
        'destaque': '2026-01',
        'filtros': {'modalidade':'Empréstimos','submodalidade':'Capital de giro com prazo de vencimento até 365 dias','segmento':'Banco','cnae_ocupacao':'Atividades financeiras, de seguros e serviços relacionados','uf':'SP','origem':'Sem destinação específica','porte':'Micro'}
    },
    '03_reversao_porte_mar2026': {
        'dimensao': 'porte',
        'titulo': 'Alteração e reversão de concentração por porte — março e abril de 2026',
        'destaque': '2026-03',
        'filtros': {'modalidade':'Empréstimos','submodalidade':'Capital de giro com prazo de vencimento até 365 dias','segmento':'Banco','cnae_ocupacao':'Atividades financeiras, de seguros e serviços relacionados','uf':'SP','origem':'Sem destinação específica','indexador':'Flutuantes'}
    },
    '04_variacao_indexador_mai2026': {
        'dimensao': 'indexador',
        'titulo': 'Variação entre indexadores — maio de 2026',
        'destaque': '2026-05',
        'filtros': {'modalidade':'Financiamentos','submodalidade':'Financiamento de projeto','segmento':'Banco','cnae_ocupacao':'Transporte, armazenagem e correio','uf':'AM','origem':'Com destinação específica','porte':'Grande'}
    }
}

def gerar_estudo(base, identificador, config):
    recorte = base
    for coluna, valor in config['filtros'].items():
        recorte = recorte.loc[recorte[coluna].astype(str).str.strip().eq(valor)]
    if recorte.empty:
        print(f'ATENÇÃO: {identificador} sem registros. Verifique a grafia dos filtros e a versão dos dados.')
        return None
    dim = config['dimensao']
    serie = (recorte.groupby([dim,'periodo'], observed=True, dropna=False)['carteira_ativa']
             .sum().unstack('periodo') / 1e9)
    meses = pd.period_range(MES_INICIAL, MES_FINAL, freq='M')
    serie = serie.reindex(columns=meses)
    serie.columns = serie.columns.astype(str)
    serie.index.name = dim
    serie.to_csv(SAIDA / f'{identificador}_saldos_mensais_bilhoes.csv', encoding='utf-8-sig', float_format='%.9f')
    print('\n' + config['titulo'])
    display(serie.round(3))
    fig, ax = plt.subplots(figsize=(13, 5.5))
    for categoria, linha in serie.iterrows():
        ax.plot(range(len(serie.columns)), linha.to_numpy(dtype=float, na_value=float('nan')), marker='o', markersize=3.5, linewidth=1.8, label=str(categoria))
    if config['destaque'] in serie.columns:
        ax.axvline(list(serie.columns).index(config['destaque']), linestyle='--', color='gray', alpha=.7)
    ax.set_title(config['titulo'], fontsize=13)
    ax.set_xlabel('Mês de referência')
    ax.set_ylabel('Carteira ativa (R$ bilhões)')
    ax.set_xticks(range(len(serie.columns)))
    ax.set_xticklabels(serie.columns, rotation=60, ha='right', fontsize=8)
    ax.grid(alpha=.18)
    ax.legend(title=dim.capitalize(), bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.text(.01,.01,'Fonte: SCR.data / BCB. Recorte agregado; variações não comprovam erros ou transferências contratuais.',fontsize=8,color='#555555')
    fig.tight_layout(rect=(0,.04,.84,1))
    fig.savefig(SAIDA / f'{identificador}_evolucao.png', dpi=170, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return serie

series_estudos = {}
for identificador, configuracao in ESTUDOS_DE_CASO.items():
    series_estudos[identificador] = gerar_estudo(base_granular, identificador, configuracao)


## 9. Exportação, parâmetros e rastreabilidade

Os arquivos gerados são reunidos em um pacote ZIP com nomenclatura neutra, adequada a repositórios públicos. Inclui tabelas de alertas, parâmetros e, quando disponíveis, tabelas e gráficos dos estudos de caso. A presença de um alerta não implica erro no dado de origem.


In [ ]:
parametros={'periodo_inicial':MES_INICIAL,'periodo_final':MES_FINAL,'cliente':'PJ',
 'valor_compensado_minimo_reais':LIMIAR_VALOR,'taxa_compensacao_minima_pct':LIMIAR_COMPENSACAO,
 'participacao_concentracao_minima_pct':LIMIAR_CONCENTRACAO,
 'saldo_concentracao_minimo_reais':LIMIAR_SALDO_CONCENTRACAO,
 'politica_ausencias':'não imputar zero; comparar somente meses consecutivos com ambas categorias publicadas',
 'fonte':BASE_URL,'metodologia':'experimental, não regulatória'}
(SAIDA/'parametros_metodologia.json').write_text(json.dumps(parametros,ensure_ascii=False,indent=2),encoding='utf-8')
with zipfile.ZipFile(PASTA/'scrdata_monitoramento_temporal_resultados.zip','w',zipfile.ZIP_DEFLATED) as z:
    for arquivo in SAIDA.glob('*'):
        if arquivo.is_file(): z.write(arquivo,arcname=arquivo.name)
print('Resultados:',(PASTA/'scrdata_monitoramento_temporal_resultados.zip').resolve())
print('Arquivos:');print('\n'.join(' • '+p.name for p in sorted(SAIDA.glob('*'))))
# Colab: descomente para baixar os resultados no computador
# from google.colab import files
# files.download(str(PASTA/'scrdata_monitoramento_temporal_resultados.zip'))

## 10. Limitações e condições de uso

- Os dados são públicos e agregados por segmento, sem identificação de contratos ou instituições individuais.
- Movimentos compensatórios e reversões não comprovam transferência dos mesmos contratos nem inconsistências na fonte.
- Uma categoria não publicada em determinado mês não é automaticamente equivalente a saldo zero.
- Os limiares são exploratórios e não representam regras de validação do Banco Central.
- Mudanças de layout, versão, cobertura ou metodologia de divulgação podem afetar a comparabilidade temporal.
- Os resultados devem ser avaliados por analistas com conhecimento do domínio antes de qualquer conclusão operacional.
